<a href="https://colab.research.google.com/github/Kosuri069/tmdb-eda-assignment/blob/main/tmdb_eda_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# tmdb_eda.ipynb

# -------------------------------
# TMDB EDA Assignment
# -------------------------------

# Task 0 — Setup
import requests
import pandas as pd
import sqlite3

# Replace 'YOUR_API_KEY_HERE' with your TMDB API key
API_KEY = "YOUR_API_KEY_HERE"  # <-- Replace this with your own TMDB API key

BASE_URL = "https://api.themoviedb.org/3/discover/movie"

# -------------------------------
# Task 1 — Build the Pipeline
# -------------------------------

# Parameters for API request
params = {
    "api_key": API_KEY,
    "language": "en-US",
    "sort_by": "popularity.desc",
    "page": 1  # pull first page
}

try:
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()  # Raise exception for bad responses
    data = response.json()
    movies = data["results"][:20]  # get first 20 movies
except requests.exceptions.RequestException as e:
    print("Error fetching TMDB data:", e)
    movies = []

# Convert genre_ids lists to strings for SQLite
df_movies["genre_ids"] = df_movies["genre_ids"].apply(lambda x: ",".join(map(str, x)))

# Save to SQLite
conn = sqlite3.connect("movies.db")
df_movies.to_sql("movies", conn, if_exists="replace", index=False)
print("Movies table saved to SQLite database.")

# -------------------------------
# Task 2 — Perform EDA
# -------------------------------

# Load movies table into DataFrame
df = pd.read_sql("SELECT * FROM movies", conn)

# 1️⃣ Display first 5 rows
print("First 5 rows:")
display(df.head())

# 2️⃣ Summary statistics for numeric columns
print("Summary statistics:")
display(df.describe())

# 3️⃣ Count number of movies per genre
# Note: TMDB returns genre_ids as list; we need to flatten it
from collections import Counter

all_genres = []
for ids in df["genre_ids"]:
    all_genres.extend(ids)  # extend with list of genre ids

genre_counts = Counter(all_genres)
print("Number of movies per genre (by genre_id):")
print(dict(genre_counts))

# 4️⃣ Identify columns with missing values
print("Missing values per column:")
print(df.isnull().sum())

Movies table saved to SQLite database.
First 5 rows:


,id,title,original_title,release_date,vote_average,vote_count,genre_ids,popularity
0,1523145,Your Heart Will Be Broken,Твое сердце будет разбито,2026-03-26,6.000,10,"1,0,7,4,9,,,1,8",718.2018
1,875828,Peaky Blinders: The Immortal Man,Peaky Blinders: The Immortal Man,2026-03-05,7.363,476,"8,0,,,1,8",290.8743
2,83533,Avatar: Fire and Ash,Avatar: Fire and Ash,2025-12-17,7.260,1946,"8,7,8,,,1,2,,,1,4",304.5720
3,687163,Project Hail Mary,Project Hail Mary,2026-03-15,8.183,587,"8,7,8,,,1,2",298.5383
4,1290821,Shelter,Shelter,2026-01-28,6.733,410,"2,8,,,8,0,,,5,3",291.7651


Summary statistics:


,id,vote_average,vote_count,popularity
count,2.000000e+01,20.000000,20.000000,20.000000
mean,1.135781e+06,6.974200,564.450000,227.072810
std,3.272576e+05,0.676037,631.317626,136.226495
min,8.353300e+04,5.800000,10.000000,105.917100
25%,1.084228e+06,6.400000,151.750000,125.465700
50%,1.196248e+06,7.055000,443.000000,222.777200
75%,1.302404e+06,7.423750,634.500000,282.942150
max,1.582770e+06,8.183000,2362.000000,718.201800


Number of movies per genre (by genre_id):
{'1': 21, ',': 200, '0': 10, '7': 17, '4': 6, '9': 5, '8': 31, '2': 17, '5': 17, '3': 14, '6': 6}
Missing values per column:
id                0
title             0
original_title    0
release_date      0
vote_average      0
vote_count        0
genre_ids         0
popularity        0
dtype: int64
